# GA Mutation Operators (Multi-Vehicle Routing)

Este notebook demonstra os **4 operadores de mutação** usados na implementação do GA deste projeto:

1. **Insertion Mutation** — remove um nó de uma posição e o insere em outra (pode cruzar veículos)
2. **Inversion Mutation** — inverte a sub-sequência entre dois índices numa rota
3. **Two-Opt Mutation** — mesma operação de reversão com motivação geométrica de 2-opt local search
4. **Swap And Redistribute** — combina dois sub-operadores: redistribuição entre veículos e troca de posições numa rota

**Interface pública:** todos os operadores implementam `mutate(solution: Individual, mutation_probability: float) -> Individual`.  
O parâmetro `mutation_probability` controla a chance de ocorrer a mutação; nos exemplos abaixo usamos `1.0` para garantir resultado visível.

Cada seção segue a estrutura:
- **Teoria** — descrição do algoritmo
- **Bare-metal** — implementação explícita em listas puras para visualizar a lógica
- **Uso com a classe do repositório** — chamada à estratégia concreta da infraestrutura

In [ ]:
#!pip install ipykernel
# Setup básico para os exemplos
import sys
from pathlib import Path

# Garantir que o root do repositório esteja no sys.path para importar o pacote src
# (o notebook fica em /concepts, então subimos um nível para o root do projeto)
sys.path.insert(0, str(Path("..").resolve()))

from src.infrastructure.genetic_algorithm.mutation import (
    InsertionMutationStrategy,
    InversionMutationStrategy,
    TwoOptMutationStrategy,
    SwapAndRedistributeMutationStrategy,
)
from src.domain.models.geo_graph.route_node import RouteNode

# Origem comum (não participa da mutação)
origin = RouteNode(name="Origin", node_id=0, coords=(0.0, 0.0))

# Destinos de exemplo — 1 veículo com 8 nós
A = RouteNode(name="A", node_id=1, coords=(1.0, 0.0))
B = RouteNode(name="B", node_id=2, coords=(2.0, 0.0))
C = RouteNode(name="C", node_id=3, coords=(2.0, 1.0))
D = RouteNode(name="D", node_id=4, coords=(1.0, 1.0))
E = RouteNode(name="E", node_id=5, coords=(0.0, 1.0))
F = RouteNode(name="F", node_id=6, coords=(0.0, 0.5))
G = RouteNode(name="G", node_id=7, coords=(1.5, 1.5))
H = RouteNode(name="H", node_id=8, coords=(2.5, 0.5))

# Individual: lista de rotas (1 veículo, com origem fixa no início)
individual = [[origin, A, B, C, D, E, F, G, H]]

def names(individual):
    return [[node.name for node in route] for route in individual]

print("Individual:", names(individual))

Individual: [['Origin', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']]


## 1) Insertion Mutation

A **Insertion Mutation** (também chamada de *Relocate*) é a operação mais direta de perturbação pontual numa permutação:

1. **Seleção:** Escolhe aleatoriamente uma rota com pelo menos 1 destino e remove um nó dela.
2. **Relocação:** Insere o nó removido numa posição aleatória de qualquer rota (pode ser a mesma rota de origem ou outra vehicle route).

Em cenários multi-veículo isso implica redistribuição de carga: um nó pode migrar do veículo A para o veículo B.  
Num cenário de veículo único, o efeito é equivalente a um *shift*: o nó sai da posição `i` e entra na posição `j ≠ i`.

**Quando usar:** Ideal para fase de exploração (alta diversidade), especialmente quando a alocação de destinos entre veículos ainda está longe do ótimo.

**Limitação:** Perturbação alta — pode destruir bons segmentos locais.

In [ ]:
# A. Implementação "Bare-metal"
def pure_insertion_mutation(route):
    """Remove um nó de uma posição e insere em outra (mesma rota, single-vehicle)."""
    route = list(route)  # cópia para não modificar o original
    
    # Posições fixas para tornar o log reproduzível
    remove_idx = 2   # vamos remover o elemento no índice 2
    insert_idx = 5   # e inserir no índice 5 (após o shift)
    
    removed = route.pop(remove_idx)
    print(f"  Nó removido: '{removed}' (era índice {remove_idx})")
    print(f"  Rota após remoção: {route}")
    
    route.insert(insert_idx, removed)
    print(f"  Nó '{removed}' reinserido em índice {insert_idx}")
    return route

sim_route = ["A", "B", "C", "D", "E", "F", "G", "H"]

print(f"Rota original:  {sim_route}")
sim_after_insertion = pure_insertion_mutation(sim_route)
print(f"Rota mutada:    {sim_after_insertion}")

Rota original:  ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
  Nó removido: 'C' (era índice 2)
  Rota após remoção: ['A', 'B', 'D', 'E', 'F', 'G', 'H']
  Nó 'C' reinserido em índice 5
Rota mutada:    ['A', 'B', 'D', 'E', 'F', 'C', 'G', 'H']


---
### Uso com `InsertionMutationStrategy` no repositório

A classe de domínio aplica o mesmo mecanismo sobre instâncias de `RouteNode`, respeitando a posição fixa da origem no índice 0 de cada rota e suportando redistribuição real entre múltiplos veículos.

In [3]:
# B. Usando a classe de domínio do projeto
insertion = InsertionMutationStrategy()

# mutation_probability=1.0 garante que a mutação sempre ocorre neste exemplo
mutated_insertion = insertion.mutate(individual, mutation_probability=1.0)

print("Original:  ", names(individual))
print("Mutado:    ", names(mutated_insertion))

Original:   [['Origin', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']]
Mutado:     [['Origin', 'H', 'A', 'B', 'C', 'D', 'E', 'F', 'G']]


## 2) Inversion Mutation

A **Inversion Mutation** (também chamada de *Reverse Sequence*) é um operador de diversificação local:

1. **Seleção de segmento:** Escolhe dois índices aleatórios `start` e `end` dentro da rota (excluindo a origem no índice 0).
2. **Reversão:** Inverte a sub-sequência `route[start:end+1]` no lugar.

O resultado é que todos os nós do segmento trocam de posição em relação ao seu espelho — o nó mais à esquerda vai para o mais à direita e vice-versa.

```
Antes:  [Origin, A, B, C, D, E, F, G, H]
Seg:    [              C, D, E         ]   (índices 3-5)
Depois: [Origin, A, B, E, D, C, F, G, H]
```

**Quando usar:** Útil para explorar vizinhanças de ordem sem alterar a composição de nós da rota. Tem baixo custo computacional e perturbação controlada.

**Limitação:** Opera em apenas uma rota por chamada; não redistribui carga entre veículos.

In [5]:
# A. Implementação "Bare-metal"
def pure_inversion_mutation(route):
    """Reverte a sub-sequência entre dois índices fixos."""
    route = list(route)  # cópia
    
    # Índices fixos para o exemplo (em produção são aleatórios)
    start, end = 2, 5  # inverte índices 2, 3, 4, 5 (inclusive)
    
    print(f"  Segmento selecionado (índices {start} a {end}): {route[start:end+1]}")
    route[start:end+1] = reversed(route[start:end+1])
    print(f"  Segmento após reversão:               {route[start:end+1]}")
    return route

sim_route = ["A", "B", "C", "D", "E", "F", "G", "H"]

print(f"Rota original:  {sim_route}")
sim_after_inversion = pure_inversion_mutation(sim_route)
print(f"Rota mutada:    {sim_after_inversion}")

Rota original:  ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
  Segmento selecionado (índices 2 a 5): ['C', 'D', 'E', 'F']
  Segmento após reversão:               ['F', 'E', 'D', 'C']
Rota mutada:    ['A', 'B', 'F', 'E', 'D', 'C', 'G', 'H']


---
### Uso com `InversionMutationStrategy` no repositório

A classe aplica a mesma lógica de reversão sobre objetos `RouteNode`. Requer rotas com mais de 2 destinos para que exista ao menos uma sub-sequência válida para inverter.

In [6]:
# B. Usando a classe de domínio do projeto
inversion = InversionMutationStrategy()

mutated_inversion = inversion.mutate(individual, mutation_probability=1.0)

print("Original:  ", names(individual))
print("Mutado:    ", names(mutated_inversion))

Original:   [['Origin', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']]
Mutado:     [['Origin', 'B', 'A', 'C', 'D', 'E', 'F', 'G', 'H']]


## 3) Two-Opt Mutation

O **Two-Opt Mutation** aplica **a mesma operação de reversão** que a Inversion Mutation, mas com uma motivação diferente: rotas do mundo real podem conter **arestas que se cruzam**, e um cruzamento quase sempre indica que a rota pode ser encurtada eliminando esse cruzamento.

### Intuição geométrica do 2-opt

Imagine a rota como um caminho no plano:
```
A → B → C → D → E → F
```
Se as arestas `(B→C)` e `(D→E)` se cruzam ao ser plotadas, é possível melhorar o percurso revertendo o trecho entre C e D:
```
A → B → D → C → E → F   (B→D e C→E não se cruzam)
```
Esse é exatamente o que `route[start:end+1] = reversed(...)` faz: tira cruzamentos.

### Diferença conceitual vs. Inversion Mutation

| Aspecto | Inversion Mutation | Two-Opt Mutation |
|---|---|---|
| Operação | Reversão de segmento | Reversão de segmento |
| Código base | Idêntico | Idêntico |
| Motivação | Diversidade / exploração | Redução de cruzamentos / explotação |
| Papel no GA | Perturbação aleatória | Refinamento local inspirado em 2-opt TSP |

**Na prática**, a implementação das duas classes é idêntica — a distinção é semântica e serve para comunicar intenção no design do GA.

In [9]:
# A. Implementação "Bare-metal"
# O código é estruturalmente idêntico à Inversion Mutation.
# O comentário mostra a intuição geométrica de remoção de cruzamentos.
def pure_two_opt_mutation(route):
    """
    Reverte route[start:end+1].
    Motivação: se as arestas (route[start-1]→route[start]) e
               (route[end]→route[end+1]) se cruzam no plano,
    a reversão as torna não-cruzantes, reduzindo o custo total.
    """
    route = list(route)
    
    # Índices fixos: simula o corte 2-opt entre as arestas B→C e E→F
    start, end = 2, 5
    
    print(f"  Aresta removida 1: {route[start-1]} → {route[start]}")
    print(f"  Aresta removida 2: {route[end]} → {route[end+1] if end+1 < len(route) else '(fim)'}")
    print(f"  Segmento a reverter: {route[start:end+1]}")
    
    route[start:end+1] = reversed(route[start:end+1])
    
    print(f"  Nova aresta 1: {route[start-1]} → {route[start]}")
    print(f"  Nova aresta 2: {route[end]} → {route[end+1] if end+1 < len(route) else '(fim)'}")
    return route

sim_route = ["A", "B", "C", "D", "E", "F", "G", "H"]

print(f"Rota original:  {sim_route}")
sim_after_twoopt = pure_two_opt_mutation(sim_route)
print(f"Rota mutada:    {sim_after_twoopt}")

Rota original:  ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
  Aresta removida 1: B → C
  Aresta removida 2: F → G
  Segmento a reverter: ['C', 'D', 'E', 'F']
  Nova aresta 1: B → F
  Nova aresta 2: C → G
Rota mutada:    ['A', 'B', 'F', 'E', 'D', 'C', 'G', 'H']


---
### Uso com `TwoOptMutationStrategy` no repositório

Idêntica à `InversionMutationStrategy` em implementação. A distinção semântica é intencional no design: quando se quer sinalizar refinamento local baseado em 2-opt local search, usa-se `TwoOptMutationStrategy`.

In [10]:
# B. Usando a classe de domínio do projeto
two_opt = TwoOptMutationStrategy()

mutated_two_opt = two_opt.mutate(individual, mutation_probability=1.0)

print("Original:  ", names(individual))
print("Mutado:    ", names(mutated_two_opt))

Original:   [['Origin', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']]
Mutado:     [['Origin', 'A', 'B', 'G', 'F', 'E', 'D', 'C', 'H']]


## 4) Swap And Redistribute Mutation

O **Swap And Redistribute** é o operador mais agressivo do conjunto: ele combina **dois sub-operadores independentes** e aplica 1 ou ambos por chamada.

### Sub-operador A: `_mutate_distribution` (redistribuição entre veículos)

1. Escolhe uma rota **fonte** com mais de 1 destino.
2. Escolhe uma rota **alvo** diferente da fonte.
3. Remove um destino aleatório da rota fonte.
4. Insere-o em posição aleatória na rota alvo.

Esse é o único sub-operador que **move nós entre veículos**.

### Sub-operador B: `_mutate_vehicle_order` (troca de posições)

1. Escolhe uma rota com pelo menos 2 destinos (tamanho > 2 contando a origem).
2. Seleciona dois índices diferentes (excluindo a origem no índice 0).
3. Faz `route[i], route[j] = route[j], route[i]`.

Esse sub-operador embaralha a **ordem de visita** dentro de um único veículo.

### Comportamento combinado

A classe sorteia aleatoriamente quantas operações aplicar (1 ou 2) e em que ordem `[distribution, vehicle_order]`. Isso torna o operador mais imprevisível — útil para escapar de ótimos locais em fases avançadas da evolução.

**Atenção com 1 veículo:** O sub-operador `distribution` exige ao menos 2 rotas; em cenário single-vehicle ele nunca é acionado. Apenas `vehicle_order` opera.

In [11]:
# A. Implementação "Bare-metal"
# Para ilustrar ambos os sub-operadores usamos um exemplo com 2 rotas de strings.

def sub_op_distribution(routes):
    """Move um nó de uma rota para outra (cross-vehicle redistribution)."""
    routes = [list(r) for r in routes]  # cópia profunda
    
    source_idx, target_idx = 0, 1          # rota 0 → rota 1
    remove_pos = 2                          # posição fixa para o log
    insert_pos = 2                          # posição de inserção na rota alvo
    
    removed = routes[source_idx].pop(remove_pos)
    print(f"  [distribution] Removido '{removed}' da Rota {source_idx} (pos {remove_pos})")
    routes[target_idx].insert(insert_pos, removed)
    print(f"  [distribution] Inserido '{removed}' na Rota {target_idx} (pos {insert_pos})")
    return routes


def sub_op_vehicle_order(routes):
    """Troca dois nós de posição dentro de uma mesma rota."""
    routes = [list(r) for r in routes]  # cópia profunda
    
    route_idx = 0   # opera na rota 0
    i, j = 1, 3     # índices fixos (ambos excluem posição 0 que seria a origem)
    
    print(f"  [vehicle_order] Swap na Rota {route_idx}: '{routes[route_idx][i]}' (pos {i}) ↔ '{routes[route_idx][j]}' (pos {j})")
    routes[route_idx][i], routes[route_idx][j] = routes[route_idx][j], routes[route_idx][i]
    return routes


# Exemplo com 2 rotas (permite demonstrar redistribuição cross-vehicle)
sim_routes = [
    ["A", "B", "C", "D"],   # Rota 0 (veículo 1)
    ["E", "F", "G", "H"],   # Rota 1 (veículo 2)
]

print("Rotas originais:")
for i, r in enumerate(sim_routes):
    print(f"  Rota {i}: {r}")

print("\n--- Sub-operador A: distribution ---")
after_distribution = sub_op_distribution(sim_routes)
for i, r in enumerate(after_distribution):
    print(f"  Rota {i}: {r}")

print("\n--- Sub-operador B: vehicle_order ---")
after_vehicle_order = sub_op_vehicle_order(sim_routes)
for i, r in enumerate(after_vehicle_order):
    print(f"  Rota {i}: {r}")

Rotas originais:
  Rota 0: ['A', 'B', 'C', 'D']
  Rota 1: ['E', 'F', 'G', 'H']

--- Sub-operador A: distribution ---
  [distribution] Removido 'C' da Rota 0 (pos 2)
  [distribution] Inserido 'C' na Rota 1 (pos 2)
  Rota 0: ['A', 'B', 'D']
  Rota 1: ['E', 'F', 'C', 'G', 'H']

--- Sub-operador B: vehicle_order ---
  [vehicle_order] Swap na Rota 0: 'B' (pos 1) ↔ 'D' (pos 3)
  Rota 0: ['A', 'D', 'C', 'B']
  Rota 1: ['E', 'F', 'G', 'H']


---
### Uso com `SwapAndRedistributeMutationStrategy` no repositório

Com o `individual` de 1 veículo definido na célula de setup, **apenas `vehicle_order` será ativado** — não há uma segunda rota para redistribuir. Para ver `distribution` em ação, seria necessário um `individual` com 2 ou mais rotas.

In [13]:
# B. Usando a classe de domínio do projeto
swap_redistribute = SwapAndRedistributeMutationStrategy()

# Com 1 veículo apenas vehicle_order sub-op dispara (distribution requer >= 2 rotas)
mutated_swap = swap_redistribute.mutate(individual, mutation_probability=1.0)

print("Original:  ", names(individual))
print("Mutado:    ", names(mutated_swap))

Original:   [['Origin', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']]
Mutado:     [['Origin', 'A', 'B', 'C', 'D', 'E', 'H', 'G', 'F']]
